In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import requests


In [ ]:
ano = 2022

In [ ]:
data_dir = Path("data")
INDIR = Path(f"../data/data_model/{ano}")
OUTDIR_IMG = Path(f"../report/img/{ano}")
OUTDIR_IMG.mkdir(parents=True, exist_ok=True)

In [ ]:
arquivo_municipio = INDIR / f"ANALISE_NOTAS_ENEM_MUNICIPIOS_BRASIL_CLUSTERS_{ano}.csv"
df_municipio = pd.read_csv(arquivo_municipio, sep=",")

In [ ]:
df_municipio.head()

In [ ]:
dict_municipio = {
    0: df_municipio[df_municipio['CLUSTER'] == 0].copy().reset_index(drop=True),
    1: df_municipio[df_municipio['CLUSTER'] == 1].copy().reset_index(drop=True),
    2: df_municipio[df_municipio['CLUSTER'] == 2].copy().reset_index(drop=True)
}

In [ ]:
renda_col = "RENDA_FAMILIAR_SM_MEDIA"

analise_notas = ['NOTA_CN_MEDIA', 'NOTA_CH_MEDIA', 'NOTA_LC_MEDIA', 'NOTA_MT_MEDIA', 'NOTA_REDACAO_MEDIA']

mapa_areas = {
    'CN': 'Ciências da Natureza e suas Tecnologias',
    'MT': 'Matemática e suas Tecnologias',
    'CH': 'Ciências Humanas e suas Tecnologias',
    'LC': 'Linguagens, Códigos e suas Tecnologias',
    'REDACAO': 'Redação'
}

colunas_notas = [c for c in analise_notas if c in df_municipio.columns]
cores_cluster = {0: "#ff0e0e", 1: "#1f77b4", 2: "#2ca02c"}

desempenho_labels = {0: "Baixo", 1: "Intermediário", 2: "Alto"}
cores_desempenho = {
    "Baixo": cores_cluster[0],
    "Intermediário": cores_cluster[1],
    "Alto": cores_cluster[2],
}

In [ ]:
colunas_plot = colunas_notas.copy()
if "NOTA_GERAL_MEDIA" in df_municipio.columns and "NOTA_GERAL_MEDIA" not in colunas_plot:
    colunas_plot.append("NOTA_GERAL_MEDIA")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, coluna in enumerate(colunas_plot):
    ax = axes[i]
    for cluster in sorted(df_municipio["CLUSTER"].unique()):
        desempenho = desempenho_labels.get(cluster, f"Cluster {cluster}")
        ax.hist(
            df_municipio.loc[df_municipio["CLUSTER"] == cluster, coluna],
            bins=20,
            alpha=0.55,
            label=desempenho,
            color=cores_desempenho.get(desempenho),
            density=True
        )

    if coluna == "NOTA_GERAL_MEDIA":
        titulo = "Média Geral"
    else:
        area_sigla = coluna.replace("NOTA_", "").replace("_MEDIA", "")
        titulo = mapa_areas.get(area_sigla, area_sigla)

    ax.set_title(titulo)
    ax.set_xlabel("Nota")
    ax.set_ylabel("Frequência")
    ax.legend()

for j in range(len(colunas_plot), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Distribuição das Médias de Notas por Cluster", fontsize=14)
plt.show()

In [ ]:
colunas_plot = colunas_notas.copy()
if "NOTA_GERAL_MEDIA" in df_municipio.columns and "NOTA_GERAL_MEDIA" not in colunas_plot:
    colunas_plot.append("NOTA_GERAL_MEDIA")

titulos = []
for c in colunas_plot:
    if c == "NOTA_GERAL_MEDIA":
        area_nome = "Média Geral"
    else:
        area_sigla = c.replace("NOTA_", "").replace("_MEDIA", "")
        area_nome = mapa_areas.get(area_sigla, area_sigla)

    d = df_municipio[[renda_col, c]].dropna()
    corr = d[renda_col].corr(d[c])
    titulos.append(f"{area_nome}<br>(Correlação: {corr:.3f})")

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=titulos,
    horizontal_spacing=0.08,
    vertical_spacing=0.25
)

for i, coluna in enumerate(colunas_plot):
    row = i // 3 + 1
    col = i % 3 + 1

    for cluster in sorted(df_municipio["CLUSTER"].unique()):
        desempenho = desempenho_labels.get(cluster, f"Cluster {cluster}")
        dados = df_municipio[df_municipio["CLUSTER"] == cluster]
        dados_validos = dados[[renda_col, coluna, "MUNICIPIO", "UF", "QTD_PARTICIPANTES"]].dropna()

        fig.add_trace(
            go.Scatter(
                x=dados_validos[renda_col],
                y=dados_validos[coluna],
                mode="markers",
                marker=dict(size=6, color=cores_desempenho.get(desempenho)),
                opacity=0.7,
                name=desempenho,
                showlegend=(i == 0),
                customdata=dados_validos[["MUNICIPIO", "UF", "QTD_PARTICIPANTES"]],
                hovertemplate=(
                    "Município: %{customdata[0]} (%{customdata[1]})<br>"
                    "Renda: %{x:.2f}<br>"
                    "Nota: %{y:.2f}<br>"
                    "Participantes: %{customdata[2]}<br>"
                    f"Desempenho: {desempenho}"
                    "<extra></extra>"
                )
            ),
            row=row,
            col=col
        )

    fig.update_xaxes(title_text="Renda Familiar Média (SM)", row=row, col=col)
    fig.update_yaxes(title_text="Nota Média", row=row, col=col)

fig.update_layout(
    title=f"Relação entre Renda Familiar Média e Notas por Desempenho (ENEM {ano})",
    height=800,
    width=1200,
    template="plotly_white",
    legend_title_text="Desempenho" 
 )

fig.show()

In [ ]:
url = "https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-100-mun.json"
geojson = requests.get(url).json()

In [ ]:
df_map = df_municipio.copy()

df_map["COD_MUNICIPIO"] = df_map["COD_MUNICIPIO"].astype(str)
df_map["DESEMPENHO"] = pd.Categorical(
    df_map["CLUSTER"].map(desempenho_labels),
    ordered=True,
    )

In [ ]:
fig = px.choropleth(
    df_map,
    geojson=geojson,
    locations="COD_MUNICIPIO",
    featureidkey="properties.id",
    color="DESEMPENHO",
    color_discrete_map=cores_desempenho,
    title=f"Desempenho por município (Brasil - ENEM {ano})",
    custom_data=[
        "MUNICIPIO",
        "UF",
        "DESEMPENHO",
        "NOTA_GERAL_MEDIA",
        "RENDA_FAMILIAR_SM_MEDIA",
        "QTD_PARTICIPANTES"
    ]
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_traces(
    hovertemplate=(
        "Município: %{customdata[0]}<br>"
        "UF: %{customdata[1]}<br>"
        "Desempenho: %{customdata[2]}<br>"
        "Média Geral: %{customdata[3]:.2f}<br>"
        "Renda média: %{customdata[4]:.2f}<br>"
        "Participantes: %{customdata[5]}<br>"
        "<extra></extra>"
    )
)

fig.show()

In [ ]:
uf_selecionada = "RS"

In [ ]:
df_map_uf = df_map.copy()
df_map_uf = df_map[df_map["UF"] == uf_selecionada]

df_map_uf["COD_MUNICIPIO"] = df_map_uf["COD_MUNICIPIO"].astype(str).str.zfill(7)

fig = px.choropleth(
    df_map_uf,
    geojson=geojson,
    locations="COD_MUNICIPIO",
    featureidkey="properties.id",
    color="DESEMPENHO",
    color_discrete_map=cores_desempenho,
    title=f"Desempenho por município ({uf_selecionada} - ENEM {ano})",
    custom_data=[
        "MUNICIPIO",
        "UF",
        "DESEMPENHO",
        "NOTA_GERAL_MEDIA",
        "RENDA_FAMILIAR_SM_MEDIA",
        "QTD_PARTICIPANTES"
    ]
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_traces(
    hovertemplate=(
        "Município: %{customdata[0]}<br>"
        "UF: %{customdata[1]}<br>"
        "Desempenho: %{customdata[2]}<br>"
        "Média Geral: %{customdata[3]:.2f}<br>"
        "Renda média: %{customdata[4]:.2f}<br>"
        "Participantes: %{customdata[5]}<br>"
        "<extra></extra>"
    )
)

fig.show()